# PHANTOM 3B Training - Google Colab

Train your own PHANTOM 3B cybersecurity assistant LLM

**Runtime type: GPU (T4 preferred)**
---

In [ ]:
# @title ## Setup
# @markdown Mount Drive and clone repo
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!git clone https://github.com/Njap-png/Cogitrongit.git
%cd Cogitrongit

In [ ]:
# @title Install Dependencies
# @markdown Install required packages
!pip install -q torch transformers accelerate datasets peft unsloth scipy bitsandbytes wandb

In [ ]:
# @title Fine-tune PHANTOM 3B
# @markdown Train the 3B model with cybersecurity knowledge
# @markdown ## ⚠️ CHANGE RUNTIME TO GPU FIRST

import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

MODEL_NAME = "meta-llama/Llama-3.1-3B-Instruct"
OUTPUT_DIR = "/content/drive/MyDrive/phantom_model"

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Load training data
dataset = load_dataset("json", data_files="data/phantom_train.jsonl")["train"]

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=2048)

dataset = dataset.map(tokenize, batched=True)

# Training
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    warmup_steps=100,
    report_to="none",
)

trainer = Trainer(model=model, args=training_args, train_dataset=dataset)
print("Training PHANTOM 3B...")
trainer.train()

# Save
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

In [ ]:
# @title Export to GGUF (for Ollama)
# @markdown Convert to GGUF format

!pip install -q llama.cpp
!python -m llama_cpp /content/drive/MyDrive/phantom_model --quantize

In [ ]:
# @title Test PHANTOM
# @markdown Test the trained model

from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    "/content/drive/MyDrive/phantom_model",
    device_map="auto",
    torch_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/phantom_model")

prompt = "What is SQL injection?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7)
print(tokenizer.decode(outputs[0]))

---
## Usage After Training

1. Download model from Drive
2. Convert to Ollama: `ollama create phantom`
3. Or use GGUF with llama.cpp
4. Run: `python phantom.py`